In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('All60Kdataset.csv')
df.head()

,serial,Headline,Content,Label
0,0,"হট্টগোল করায় বাকৃবিতে দুইজন বরখাস্ত, ৬ জনকে শোকজ",গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে ...,1
1,1,মালয়েশিয়ায় কর্মী পাঠানোর ব্যবস্থা নেয়ার সুপারিশ,বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রম...,1
2,2,প্রেমের প্রস্তাবে রাজি না হওয়ায় স্কুলছাত্রীকে ...,নরসিংদীর মনোহরদীতে প্রেমের প্রস্তাবে রাজি না হ...,1
3,3,মেডিয়েশনই মামলাজট নিরসনের পথ : বিচারপতি আহমেদ ...,সুপ্রিম কোর্টের হাইকোর্ট বিভাগের বিচারপতি আহমে...,1
4,4,টকশোতে বক্তব্য দিতে গিয়ে জাপা নেতার মৃত্যু,মাদারীপুর সদরের উপজেলার লেকেরপাড়ে একটি বেসরকার...,1


In [6]:
df.nunique()

serial      61581
Headline    53813
Content     58394
Label           2
dtype: int64

In [3]:
#print how many element each label has
df['Label'].value_counts()

Label
1    48678
0    12903
Name: count, dtype: int64

In [5]:
df['word_count'] = df['Content'].apply(lambda x: len(x.split()))
df['word_count'].describe()

count    61581.000000
mean       276.312353
std        229.548848
min          0.000000
25%        142.000000
50%        213.000000
75%        327.000000
max       4788.000000
Name: word_count, dtype: float64

In [6]:
print(f"Empty articles: {(df['word_count'] == 0).sum()}")
df = df[df['word_count'] > 10].reset_index(drop=True)
print(f"Dataset after cleaning: {len(df)}")

Empty articles: 1
Dataset after cleaning: 61515


In [8]:
print(df['Label'].value_counts())
print(df['Label'].value_counts(normalize=True).round(3) * 100)

Label
1    48627
0    12888
Name: count, dtype: int64
Label
1    79.0
0    21.0
Name: proportion, dtype: float64


In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

# Sample 2000 articles for speed
sample_df = df.sample(2000, random_state=42)
sample_df['token_count'] = sample_df['Content'].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print(sample_df['token_count'].describe())
print(f"\nArticles > 512 tokens:  {(sample_df['token_count'] > 512).mean()*100:.1f}%")
print(f"Articles > 1024 tokens: {(sample_df['token_count'] > 1024).mean()*100:.1f}%")
print(f"Articles <= 512 tokens: {(sample_df['token_count'] <= 512).mean()*100:.1f}%")

count    2000.000000
mean      370.088500
std       305.687425
min        24.000000
25%       188.000000
50%       282.500000
75%       437.000000
max      3796.000000
Name: token_count, dtype: float64

Articles > 512 tokens:  18.4%
Articles > 1024 tokens: 3.8%
Articles <= 512 tokens: 81.6%


In [12]:
print("Word count by label:")
print(df.groupby('Label')['word_count'].describe().round(1))

Word count by label:
         count   mean    std   min    25%    50%    75%     max
Label                                                          
0      12888.0  273.3  209.2  12.0  150.0  224.0  327.0  3350.0
1      48627.0  277.5  234.6  11.0  141.0  211.0  327.0  4788.0


In [13]:
# Remove articles that are absurdly short or long
df = df[(df['word_count'] >= 20) & (df['word_count'] <= 2000)]
print(f"After length filtering: {len(df)} articles remain")

After length filtering: 61335 articles remain


In [ ]:
# Get exact percentages with real tokenizer on full dataset
df['token_count'] = df['Content'].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True, 
                                    max_length=2000, truncation=True))
)
print(f"> 512 tokens: {(df['token_count'] > 512).mean()*100:.1f}%")
print(f"> 768 tokens: {(df['token_count'] > 768).mean()*100:.1f}%")
print(f"> 1024 tokens: {(df['token_count'] > 1024).mean()*100:.1f}%")

In [1]:
import pandas as pd
long  = pd.read_csv('./long_test_subset.csv')
short = pd.read_csv('./short_test_subset.csv')
print(len(long), len(short))

1088 4713
